# Ordered Logistic Regression Results for Adoption Predictors: Exploration with `mlcroissant`

This notebook demonstrates how to load, examine, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

**Dataset source:**

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

*Note: All references to record sets, fields, and columns use their Croissant schema `@id` fields, per best practices and reproducibility requirements.*

In [ ]:
# Ensure mlcroissant is installed in your environment
!pip install mlcroissant pandas matplotlib

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

First, define the Croissant schema URL and load the `Dataset` via the `mlcroissant.Dataset` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List all available record sets, fields, and columns in the dataset, referencing each entity by its `@id`.

*For each record set, print its `@id`, and then list available field and column `@id`s.*

In [ ]:
# List record sets and their fields/columns by @id
print("Available Record Sets and Fields by @id:")

record_sets = [r for r in dataset.record_sets()]

if not record_sets:
    print('No record sets found in dataset. Please check the Croissant schema for recordSet definitions.')
else:
    for rec in record_sets:
        print(f"\nRecordSet @id: {rec['@id']}")
        if 'fields' in rec:
            for f in rec['fields']:
                if isinstance(f, dict):
                    print(f"  Field @id: {f.get('@id','<unknown>')}")
                else:
                    print(f"  Field @id: {f}")
        if 'columns' in rec:
            for c in rec['columns']:
                if isinstance(c, dict):
                    print(f"  Column @id: {c.get('@id','<unknown>')}")
                else:
                    print(f"  Column @id: {c}")

## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame`, referencing each by record set `@id`.

Below, we extract all record sets we identified in the previous cell. If you know a specific record set `@id` of interest, you can adapt the code to focus only on that set.

In [ ]:
# Discover available record set @id's for extraction
record_set_ids = [rec['@id'] for rec in dataset.record_sets()] if record_sets else []

# Display the found record set ids
print('RecordSet @ids available:', record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    print(f"\nExtracting records for RecordSet @id: {rs_id}")
    # Use dataset.records(record_set=...) with the @id
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame columns for '@id' {rs_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for RecordSet @id: {rs_id}")

if not dataframes:
    print("No tabular data extracted. Please check if the dataset contains accessible record sets.")
else:
    rs0 = list(dataframes.keys())[0]
    print(f"\nExample DataFrame columns for RecordSet @id {rs0}: {dataframes[rs0].columns.tolist()}")
    display(dataframes[rs0].head())

## 4. Exploratory Data Analysis (EDA)

Apply basic EDA on a selected record set. This includes filtering, normalization, and grouping (if suitable fields exist).

**Note:** You will need to use the actual `@id` for the record set and field(s) to proceed. For demonstration, the code selects the first available record set and tries to find a numeric field.

In [ ]:
# Pick first loaded record set if available
if dataframes:
    rec_set_id = list(dataframes.keys())[0]
    df = dataframes[rec_set_id]
    print(f"Exploring DataFrame for RecordSet @id: {rec_set_id}")
    # Find a numeric column by checking dtypes
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean()  # use mean as dynamic threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a string/categorical column (if any)
        group_field = None
        # Exclude the numeric field from available columns
        possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"\nMean {numeric_field_id} grouped by {group_field}:")
            print(grouped_df.head())
    else:
        print(f"No numeric fields found in DataFrame for RecordSet @id: {rec_set_id}")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization

Visualize the distribution of the main numeric field in the selected RecordSet, and (if a group field was available), plot means by group.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals():
    # Distribution plot
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=20, alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id} in RecordSet @id {rec_set_id}')
    plt.show()

    # Grouped bar plot if possible
    if 'grouped_df' in locals() and group_field is not None:
        grouped_df.plot(kind='bar', legend=False, figsize=(8, 4))
        plt.xlabel(group_field)
        plt.ylabel(f'mean({numeric_field_id})')
        plt.title(f'Mean {numeric_field_id} by {group_field}')
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and examine data from the FAIR^2 dataset using `mlcroissant`, performing initial exploratory analysis and simple visualizations using only Croissant `@id` references.

Key takeaways:
- The `mlcroissant` library enables direct record extraction and schema navigation via Croissant's standard `@id` references.
- The dataset schema exposes all data entities with unique `@id`s, promoting traceability and reproducibility across exploration steps.
- Initial analysis and plots can be flexibly adapted to your analytic workflow.

*For further analysis, review full dataset documentation and experiment with more advanced filtering, grouping, and visualization operations grouped by `@id` field references.*